# MLP Copy Code

This notebook contains a simple implementation of a Multi-Layer Perceptron (MLP) for character-level language modeling. The code is designed to process sequences of characters and predict the next character in the sequence.

In [2]:
import torch
import random
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
def build_dataset(words: list, stoi: dict) -> tuple:
    device = 'cuda' if torch.cuda.is_available() else "cpu"
    block_size = 3
    X, Y = [], []

    for w in words:
        content = [0]*block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(content)
            Y.append(ix)
            content = content[1:] + [ix]

    X = torch.tensor(X, device=device)
    Y = torch.tensor(Y, device=device)
    return X, Y

In [4]:
device = 'cuda' if torch.cuda.is_available() else "cpu"
f"Tensors are saved on {device}"

'Tensors are saved on cuda'

In [5]:
words = open("dataset/names.txt", 'r').read().splitlines()
f"Number of names {len(words)}"

'Number of names 32033'

In [6]:
chars = sorted(set(list(''.join(words))))
stoi = {ch: i+1 for i, ch in enumerate(chars)}
stoi['.'] = 0
itos = {i: ch for ch, i in stoi.items()}

In [7]:
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

X, Y = build_dataset(words, stoi)
Xtr, Ytr = build_dataset(words[:n1], stoi)
Xde, Yde = build_dataset(words[n1:n2], stoi)
Xte, Yte = build_dataset(words[n2:], stoi)

In [8]:
Xtr.shape, Ytr.shape

(torch.Size([182625, 3]), torch.Size([182625]))

In [9]:
C = torch.randn((27, 2)).to(device)
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [10]:
emb[1]

tensor([[-0.4822, -0.4119],
        [-0.4822, -0.4119],
        [-0.0398,  0.1873]], device='cuda:0')

In [11]:
W1 = torch.randn((6,100)).to(device)
B1 = torch.randn(100).to(device)
h = torch.tanh(emb.view(-1, 6) @ W1 + B1)

In [12]:
h.shape

torch.Size([228146, 100])

In [13]:
W2 = torch.randn((100,27)).to(device)
B2 = torch.randn(27).to(device)
logits = h @ W2 + B2

In [14]:
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)

In [15]:
prob.shape

torch.Size([228146, 27])

In [16]:
loss = -prob[torch.arange(len(Y)), Y].log().mean()
loss

tensor(15.9362, device='cuda:0')

In [17]:
Xtr.size(), Ytr.size()

(torch.Size([182625, 3]), torch.Size([182625]))

In [21]:
g = torch.Generator(device=device).manual_seed(2147483647)
C = torch.randn((27,10), generator=g, device=device)        ; C.requires_grad = True
W1 = torch.randn((30,200), generator=g, device=device)      ; W1.requires_grad = True
B1 = torch.randn(200, generator=g, device=device)           ; B1. requires_grad = True
W2 = torch.randn((200,27), generator=g, device=device)      ; W2.requires_grad = True
B1 = torch.randn(27, generator=g, device=device)            ; B2.requires_grad = True
parameters = [C, W1, B1, W2, B2]

In [22]:
stepi = []
lossi = []

for epoch in range(200000):
    batch = torch.randint(0, Xtr.shape[0], (32,))
    emb = C[Xtr[batch]]
    h = torch.tanh(emb.view(-1, 30) @ W1 + B1)
    logits = h @ W2 + B2
    loss = F.cross_entropy(logits, Ytr[batch])
    for p in parameters:
        p.grad = None
    loss.backward()

    lr = 0.1 if epoch < 100000 else 0.01
    for p in parameters:
        p -= lr*p.grad

    stepi.append(epoch)
    lossi.append(loss.log10().item())

RuntimeError: The size of tensor a (200) must match the size of tensor b (27) at non-singleton dimension 1